In [15]:

# STEP 1: Install project dependencies

!pip -q install sentence-transformers scikit-learn pandas numpy matplotlib tqdm

print("All project libraries installed successfully!")


All project libraries installed successfully!


In [16]:

# FIXED STEP 2: Download MIND-small from Hugging Face

import os
import zipfile
from pathlib import Path
from huggingface_hub import hf_hub_download

DATA_DIR = Path("/content/news_recommendation_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

datasets = {
    "train": "MINDsmall_train.zip",
    "validation": "MINDsmall_dev.zip"
}

for split, filename in datasets.items():

    print(f"\nDownloading {filename} from Hugging Face...")

    zip_path = hf_hub_download(
        repo_id="Recommenders/MIND",
        filename=filename,
        repo_type="dataset"
    )

    extract_path = DATA_DIR / split
    extract_path.mkdir(parents=True, exist_ok=True)

    print(f"Extracting {filename}...")

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_path)

    print(f"{split.upper()} dataset ready!")

print("\nSUCCESS! MIND-SMALL DOWNLOADED.")



Extracting MINDsmall_train.zip...
TRAIN dataset ready!

Extracting MINDsmall_dev.zip...
VALIDATION dataset ready!

SUCCESS! MIND-SMALL DOWNLOADED.


In [17]:

# STEP 3: Load MIND dataset into Pandas

import pandas as pd

NEWS_COLUMNS = [
    "news_id",
    "category",
    "subcategory",
    "title",
    "abstract",
    "url",
    "title_entities",
    "abstract_entities"
]

BEHAVIOR_COLUMNS = [
    "impression_id",
    "user_id",
    "timestamp",
    "history",
    "impressions"
]

def load_mind_split(split):
    folder = DATA_DIR / split

    news_file = next(folder.rglob("news.tsv"))
    behavior_file = next(folder.rglob("behaviors.tsv"))

    news = pd.read_csv(
        news_file,
        sep="\t",
        header=None,
        names=NEWS_COLUMNS,
        keep_default_na=False
    )

    behaviors = pd.read_csv(
        behavior_file,
        sep="\t",
        header=None,
        names=BEHAVIOR_COLUMNS,
        keep_default_na=False
    )

    return news, behaviors


train_news, train_behaviors = load_mind_split("train")
val_news, val_behaviors = load_mind_split("validation")

print("TRAIN NEWS:", train_news.shape)
print("TRAIN BEHAVIORS:", train_behaviors.shape)

print("\nVALIDATION NEWS:", val_news.shape)
print("VALIDATION BEHAVIORS:", val_behaviors.shape)

print("\nSAMPLE NEWS:")
display(train_news.head())

print("\nSAMPLE USER BEHAVIOURS:")
display(train_behaviors.head())


TRAIN NEWS: (51282, 8)
TRAIN BEHAVIORS: (156965, 5)

VALIDATION NEWS: (42416, 8)
VALIDATION BEHAVIORS: (73152, 5)

SAMPLE NEWS:


,news_id,category,subcategory,title,abstract,url,title_entities,abstract_entities
0,N55528,lifestyle,lifestyleroyals,"The Brands Queen Elizabeth, Prince Charles, an...","Shop the notebooks, jackets, and more that the...",https://assets.msn.com/labs/mind/AAGH0ET.html,"[{""Label"": ""Prince Philip, Duke of Edinburgh"",...",[]
1,N19639,health,weightloss,50 Worst Habits For Belly Fat,These seemingly harmless habits are holding yo...,https://assets.msn.com/labs/mind/AAB19MK.html,"[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik...","[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik..."
2,N61837,news,newsworld,The Cost of Trump's Aid Freeze in the Trenches...,Lt. Ivan Molchanets peeked over a parapet of s...,https://assets.msn.com/labs/mind/AAJgNsz.html,[],"[{""Label"": ""Ukraine"", ""Type"": ""G"", ""WikidataId..."
3,N53526,health,voices,I Was An NBA Wife. Here's How It Affected My M...,"I felt like I was a fraud, and being an NBA wi...",https://assets.msn.com/labs/mind/AACk2N6.html,[],"[{""Label"": ""National Basketball Association"", ..."
4,N38324,health,medical,"How to Get Rid of Skin Tags, According to a De...","They seem harmless, but there's a very good re...",https://assets.msn.com/labs/mind/AAAKEkt.html,"[{""Label"": ""Skin tag"", ""Type"": ""C"", ""WikidataI...","[{""Label"": ""Skin tag"", ""Type"": ""C"", ""WikidataI..."



SAMPLE USER BEHAVIOURS:


,impression_id,user_id,timestamp,history,impressions
0,1,U13740,11/11/2019 9:05:58 AM,N55189 N42782 N34694 N45794 N18445 N63302 N104...,N55689-1 N35729-0
1,2,U91836,11/12/2019 6:11:30 PM,N31739 N6072 N63045 N23979 N35656 N43353 N8129...,N20678-0 N39317-0 N58114-0 N20495-0 N42977-0 N...
2,3,U73700,11/14/2019 7:01:48 AM,N10732 N25792 N7563 N21087 N41087 N5445 N60384...,N50014-0 N23877-0 N35389-0 N49712-0 N16844-0 N...
3,4,U34670,11/11/2019 5:28:05 AM,N45729 N2203 N871 N53880 N41375 N43142 N33013 ...,N35729-0 N33632-0 N49685-1 N27581-0
4,5,U8125,11/12/2019 4:11:21 PM,N10078 N56514 N14904 N33740,N39985-0 N36050-0 N16096-0 N8400-1 N22407-0 N6...


In [18]:

# STEP 4: Clean and preprocess news data

import pandas as pd
import numpy as np
import re

# Work on copies so original datasets remain unchanged
news_df = train_news.copy()
behaviors_df = train_behaviors.copy()

# Combine title and abstract into one text field
news_df["title"] = news_df["title"].fillna("").astype(str)
news_df["abstract"] = news_df["abstract"].fillna("").astype(str)

news_df["text"] = (
    news_df["title"].str.strip() + ". " +
    news_df["abstract"].str.strip()
).str.strip()

# Remove empty articles and duplicate IDs
news_df = news_df.drop_duplicates(subset="news_id")
news_df = news_df[news_df["text"].str.len() > 5]

# Normalize category and subcategory
news_df["category"] = (
    news_df["category"].fillna("unknown")
    .astype(str).str.lower().str.strip()
)

news_df["subcategory"] = (
    news_df["subcategory"].fillna("unknown")
    .astype(str).str.lower().str.strip()
)

# Clean whitespace
news_df["text"] = news_df["text"].str.replace(
    r"\s+", " ", regex=True
).str.strip()

# Parse timestamps
behaviors_df["timestamp"] = pd.to_datetime(
    behaviors_df["timestamp"],
    errors="coerce"
)

print("DATA CLEANING COMPLETE")
print("-" * 40)

print("Clean news articles:", len(news_df))
print("Behaviour records:", len(behaviors_df))
print("Unique users:", behaviors_df["user_id"].nunique())
print("Unique categories:", news_df["category"].nunique())
print("Missing article text:", news_df["text"].isna().sum())
print("Missing timestamps:", behaviors_df["timestamp"].isna().sum())

print("\nTOP NEWS CATEGORIES:")
display(news_df["category"].value_counts().head(10))

print("\nCLEANED ARTICLE EXAMPLES:")
display(news_df[["news_id", "category", "title", "text"]].head())


DATA CLEANING COMPLETE
----------------------------------------
Clean news articles: 51282
Behaviour records: 156965
Unique users: 50000
Unique categories: 17
Missing article text: 0
Missing timestamps: 0

TOP NEWS CATEGORIES:


,count
category,
news,15774
sports,14510
finance,3107
foodanddrink,2551
lifestyle,2479
travel,2350
video,2068
weather,2048
health,1885



CLEANED ARTICLE EXAMPLES:


,news_id,category,title,text
0,N55528,lifestyle,"The Brands Queen Elizabeth, Prince Charles, an...","The Brands Queen Elizabeth, Prince Charles, an..."
1,N19639,health,50 Worst Habits For Belly Fat,50 Worst Habits For Belly Fat. These seemingly...
2,N61837,news,The Cost of Trump's Aid Freeze in the Trenches...,The Cost of Trump's Aid Freeze in the Trenches...
3,N53526,health,I Was An NBA Wife. Here's How It Affected My M...,I Was An NBA Wife. Here's How It Affected My M...
4,N38324,health,"How to Get Rid of Skin Tags, According to a De...","How to Get Rid of Skin Tags, According to a De..."


In [19]:

# STEP 5: Extract user interaction records

interactions = []

for _, row in behaviors_df.iterrows():

    user_id = row["user_id"]

    # Previously clicked articles
    history = str(row["history"]).split()

    for news_id in history:
        interactions.append({
            "user_id": user_id,
            "news_id": news_id,
            "label": 1,
            "interaction_type": "history"
        })

    # Current article impressions
    impression_items = str(row["impressions"]).split()

    for item in impression_items:

        if "-" not in item:
            continue

        news_id, label = item.rsplit("-", 1)

        if label not in ["0", "1"]:
            continue

        interactions.append({
            "user_id": user_id,
            "news_id": news_id,
            "label": int(label),
            "interaction_type": "impression"
        })


interactions_df = pd.DataFrame(interactions)

# Keep only articles available in our training news collection
valid_news_ids = set(news_df["news_id"])

interactions_df = interactions_df[
    interactions_df["news_id"].isin(valid_news_ids)
].copy()

print("INTERACTION EXTRACTION COMPLETE")
print("-" * 40)

print("Total interactions:", len(interactions_df))
print("Unique users:", interactions_df["user_id"].nunique())
print("Unique articles:", interactions_df["news_id"].nunique())

print("\nINTERACTION LABEL DISTRIBUTION:")
display(interactions_df["label"].value_counts())

print("\nINTERACTION EXAMPLES:")
display(interactions_df.head(10))


INTERACTION EXTRACTION COMPLETE
----------------------------------------
Total interactions: 10951083
Unique users: 50000
Unique articles: 51282

INTERACTION LABEL DISTRIBUTION:


,count
label,
0,5607100
1,5343983



INTERACTION EXAMPLES:


,user_id,news_id,label,interaction_type
0,U13740,N55189,1,history
1,U13740,N42782,1,history
2,U13740,N34694,1,history
3,U13740,N45794,1,history
4,U13740,N18445,1,history
5,U13740,N63302,1,history
6,U13740,N10414,1,history
7,U13740,N19347,1,history
8,U13740,N31801,1,history
9,U13740,N55689,1,impression


In [20]:

# STEP 6: Dataset quality and coverage analysis

print("DATASET QUALITY REPORT")
print("=" * 45)

print("News records:", len(news_df))
print("Interaction records:", len(interactions_df))
print("Unique users:", interactions_df["user_id"].nunique())
print("Unique articles interacted with:", interactions_df["news_id"].nunique())

positive = (interactions_df["label"] == 1).sum()
negative = (interactions_df["label"] == 0).sum()

print("\nPositive interactions:", positive)
print("Negative impressions:", negative)

if positive + negative > 0:
    print("Positive interaction ratio:",
          round(positive / (positive + negative) * 100, 2), "%")

print("\nMost active users:")
display(
    interactions_df.groupby("user_id")
    .size()
    .sort_values(ascending=False)
    .head(10)
)

print("\nMOST POPULAR ARTICLES:")
display(
    interactions_df[interactions_df["label"] == 1]
    ["news_id"].value_counts().head(10)
)

print("\nDATASET QUALITY CHECK COMPLETE!")


DATASET QUALITY REPORT
News records: 51282
Interaction records: 10951083
Unique users: 50000
Unique articles interacted with: 51282

Positive interactions: 5343983
Negative impressions: 5607100
Positive interaction ratio: 48.8 %

Most active users:


,0
user_id,
U63482,20396
U20833,16232
U79449,14730
U79210,13270
U84756,11711
U32146,10872
U72489,10472
U44210,10230
U67455,9697



MOST POPULAR ARTICLES:


,count
news_id,
N306,20765
N42620,17918
N47020,16300
N31801,15694
N45794,15200
N43142,14565
N871,14210
N55189,13394
N29177,13243



DATASET QUALITY CHECK COMPLETE!


In [21]:

# STEP 7: Build clean, deduplicated interaction datasets

# 1. Extract historical reading interactions
history_df = (
    behaviors_df[["user_id", "history"]]
    .assign(news_id=lambda x: x["history"].str.split())
    .explode("news_id")
    [["user_id", "news_id"]]
)

history_df = history_df[
    history_df["news_id"].notna() &
    history_df["news_id"].ne("")
]

# Remove repeated user-article history pairs
history_df = history_df.drop_duplicates(
    subset=["user_id", "news_id"]
).copy()

history_df["label"] = 1
history_df["interaction_type"] = "history"

# 2. Extract individual impressions
impression_df = (
    behaviors_df[["impression_id", "user_id", "impressions"]]
    .assign(item=lambda x: x["impressions"].str.split())
    .explode("item")
    .dropna(subset=["item"])
)

parts = impression_df["item"].str.rsplit("-", n=1, expand=True)

impression_df["news_id"] = parts[0]
impression_df["label"] = pd.to_numeric(
    parts[1], errors="coerce"
)

impression_df = impression_df.dropna(
    subset=["news_id", "label"]
).copy()

impression_df["label"] = impression_df["label"].astype("int8")

impression_df = impression_df[
    impression_df["label"].isin([0, 1])
]

impression_df = impression_df[
    impression_df["news_id"].isin(valid_news_ids)
].copy()

# 3. Create unique positive user-article pairs
clicked_impressions = impression_df[
    impression_df["label"] == 1
][["user_id", "news_id"]]

positive_interactions = pd.concat(
    [
        history_df[["user_id", "news_id"]],
        clicked_impressions
    ],
    ignore_index=True
).drop_duplicates()

print("CORRECTED INTERACTION DATA")
print("=" * 45)

print("Unique historical pairs:", len(history_df))
print("Individual impressions:", len(impression_df))
print("Positive impression records:",
      (impression_df["label"] == 1).sum())
print("Negative impression records:",
      (impression_df["label"] == 0).sum())
print("Unique positive user-article pairs:",
      len(positive_interactions))

print("\nSample historical interactions:")
display(history_df.head())

print("\nSample impressions:")
display(impression_df.head())

print("\nSample positive interactions:")
display(positive_interactions.head())


CORRECTED INTERACTION DATA
Unique historical pairs: 915011
Individual impressions: 5843444
Positive impression records: 236344
Negative impression records: 5607100
Unique positive user-article pairs: 1148447

Sample historical interactions:


,user_id,news_id,label,interaction_type
0,U13740,N55189,1,history
0,U13740,N42782,1,history
0,U13740,N34694,1,history
0,U13740,N45794,1,history
0,U13740,N18445,1,history



Sample impressions:


,impression_id,user_id,impressions,item,news_id,label
0,1,U13740,N55689-1 N35729-0,N55689-1,N55689,1
0,1,U13740,N55689-1 N35729-0,N35729-0,N35729,0
1,2,U91836,N20678-0 N39317-0 N58114-0 N20495-0 N42977-0 N...,N20678-0,N20678,0
1,2,U91836,N20678-0 N39317-0 N58114-0 N20495-0 N42977-0 N...,N39317-0,N39317,0
1,2,U91836,N20678-0 N39317-0 N58114-0 N20495-0 N42977-0 N...,N58114-0,N58114,0



Sample positive interactions:


,user_id,news_id
0,U13740,N55189
1,U13740,N42782
2,U13740,N34694
3,U13740,N45794
4,U13740,N18445


In [22]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Selected device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

Selected device: cuda
GPU: Tesla T4


In [23]:
!pip install -q sentence-transformers

In [24]:
from sentence_transformers import SentenceTransformer

model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

model = SentenceTransformer(
    model_name,
    device=device
)

print("Multilingual model loaded successfully!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Multilingual model loaded successfully!


In [25]:
print("Model device:", model.device)
print("Embedding dimension:", model.get_sentence_embedding_dimension())

Model device: cuda:0
Embedding dimension: 384


/tmp/ipykernel_1913/3864144540.py:2: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", model.get_sentence_embedding_dimension())


In [26]:
import os

print("Current /content contents:")
print(os.listdir("/content"))

print("\nCurrent Python variables:")
for name in ["news_df", "train_news", "model", "device"]:
    print(name, ":", name in globals())

Current /content contents:
['.config', 'drive', 'news_recommendation_data', 'sample_data']

Current Python variables:
news_df : True
train_news : True
model : True
device : True


In [27]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [28]:
import os

PROJECT_DIR = "/content/drive/MyDrive/News_Recommendation_Project"

os.makedirs(PROJECT_DIR, exist_ok=True)

print("Project folder ready:", PROJECT_DIR)
print("Model device:", model.device)

Project folder ready: /content/drive/MyDrive/News_Recommendation_Project
Model device: cuda:0


In [29]:
from huggingface_hub import HfApi

api = HfApi()

files = api.list_repo_files(
    repo_id="Recommenders/MIND",
    repo_type="dataset"
)

print("Available MIND dataset ZIP files:")

for file in files:
    if file.lower().endswith(".zip"):
        print(file)

Available MIND dataset ZIP files:
MINDdemo_dev.zip
MINDdemo_train.zip
MINDdemo_utils.zip
MINDlarge_dev.zip
MINDlarge_test.zip
MINDlarge_train.zip
MINDlarge_utils.zip
MINDsmall_dev.zip
MINDsmall_train.zip
MINDsmall_utils.zip
lstur.zip
naml.zip
npa.zip
nrms.zip


In [30]:
from huggingface_hub import hf_hub_download
from pathlib import Path
import zipfile

# Permanent Google Drive locations
PROJECT_DIR = Path(
    "/content/drive/MyDrive/News_Recommendation_Project"
)

RAW_DIR = PROJECT_DIR / "raw_data"
TRAIN_DIR = RAW_DIR / "train"
VAL_DIR = RAW_DIR / "validation"

for folder in [RAW_DIR, TRAIN_DIR, VAL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Download and extract both dataset splits
datasets = {
    "MINDsmall_train.zip": TRAIN_DIR,
    "MINDsmall_dev.zip": VAL_DIR
}

for filename, destination in datasets.items():

    print(f"\nDownloading {filename}...")

    zip_path = hf_hub_download(
        repo_id="Recommenders/MIND",
        filename=filename,
        repo_type="dataset"
    )

    print("Download complete. Extracting...")

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(destination)

    print("Extracted into:", destination)

print("\nMIND-small dataset restored successfully!")


Download complete. Extracting...
Extracted into: /content/drive/MyDrive/News_Recommendation_Project/raw_data/train

Download complete. Extracting...
Extracted into: /content/drive/MyDrive/News_Recommendation_Project/raw_data/validation

MIND-small dataset restored successfully!


In [31]:
for split, folder in [
    ("TRAIN", TRAIN_DIR),
    ("VALIDATION", VAL_DIR)
]:
    print(f"\n{split} FILES:")

    for file in folder.rglob("*.tsv"):
        print(file.name, "|", file.stat().st_size, "bytes")


TRAIN FILES:
news.tsv | 41202121 bytes
behaviors.tsv | 92019716 bytes

VALIDATION FILES:
behaviors.tsv | 42838544 bytes
news.tsv | 33519092 bytes


In [32]:
import pandas as pd
from pathlib import Path

# Dataset locations in Google Drive
PROJECT_DIR = Path(
    "/content/drive/MyDrive/News_Recommendation_Project"
)

TRAIN_DIR = PROJECT_DIR / "raw_data" / "train"
VAL_DIR = PROJECT_DIR / "raw_data" / "validation"

NEWS_COLUMNS = [
    "news_id", "category", "subcategory", "title",
    "abstract", "url", "title_entities", "abstract_entities"
]

BEHAVIOR_COLUMNS = [
    "impression_id", "user_id", "timestamp",
    "history", "impressions"
]

# Load both splits
def load_mind_split(folder):
    news_file = next(folder.rglob("news.tsv"))
    behavior_file = next(folder.rglob("behaviors.tsv"))

    news = pd.read_csv(
        news_file,
        sep="\t",
        header=None,
        names=NEWS_COLUMNS,
        keep_default_na=False
    )

    behaviors = pd.read_csv(
        behavior_file,
        sep="\t",
        header=None,
        names=BEHAVIOR_COLUMNS,
        keep_default_na=False
    )

    return news, behaviors


train_news, train_behaviors = load_mind_split(TRAIN_DIR)
val_news, val_behaviors = load_mind_split(VAL_DIR)

# Clean article data
def clean_news(news):
    news = news.copy()

    news["title"] = news["title"].fillna("").astype(str)
    news["abstract"] = news["abstract"].fillna("").astype(str)

    news["text"] = (
        news["title"].str.strip() + ". " +
        news["abstract"].str.strip()
    ).str.strip()

    news = news.drop_duplicates(subset="news_id")
    news = news[news["text"].str.len() > 5].copy()

    for column in ["category", "subcategory"]:
        news[column] = (
            news[column].fillna("unknown")
            .astype(str).str.lower().str.strip()
        )

    news["text"] = (
        news["text"].str.replace(r"\s+", " ", regex=True).str.strip()
    )

    return news


news_df = clean_news(train_news)
val_news_df = clean_news(val_news)

# Clean timestamps
train_behaviors["timestamp"] = pd.to_datetime(
    train_behaviors["timestamp"], errors="coerce"
)

val_behaviors["timestamp"] = pd.to_datetime(
    val_behaviors["timestamp"], errors="coerce"
)

# Save cleaned data permanently
news_df.to_pickle(PROJECT_DIR / "train_news_clean.pkl")
val_news_df.to_pickle(PROJECT_DIR / "validation_news_clean.pkl")

train_behaviors.to_pickle(
    PROJECT_DIR / "train_behaviors.pkl"
)

val_behaviors.to_pickle(
    PROJECT_DIR / "validation_behaviors.pkl"
)

print("DATA RESTORATION COMPLETE")
print("=" * 40)

print("TRAIN NEWS:", news_df.shape)
print("TRAIN BEHAVIORS:", train_behaviors.shape)

print("VALIDATION NEWS:", val_news_df.shape)
print("VALIDATION BEHAVIORS:", val_behaviors.shape)

print("\nSaved all four cleaned datasets to Google Drive.")

display(news_df[["news_id", "title", "category", "text"]].head())

DATA RESTORATION COMPLETE
TRAIN NEWS: (51282, 9)
TRAIN BEHAVIORS: (156965, 5)
VALIDATION NEWS: (42416, 9)
VALIDATION BEHAVIORS: (73152, 5)

Saved all four cleaned datasets to Google Drive.


,news_id,title,category,text
0,N55528,"The Brands Queen Elizabeth, Prince Charles, an...",lifestyle,"The Brands Queen Elizabeth, Prince Charles, an..."
1,N19639,50 Worst Habits For Belly Fat,health,50 Worst Habits For Belly Fat. These seemingly...
2,N61837,The Cost of Trump's Aid Freeze in the Trenches...,news,The Cost of Trump's Aid Freeze in the Trenches...
3,N53526,I Was An NBA Wife. Here's How It Affected My M...,health,I Was An NBA Wife. Here's How It Affected My M...
4,N38324,"How to Get Rid of Skin Tags, According to a De...",health,"How to Get Rid of Skin Tags, According to a De..."


In [33]:
# STEP 5: Generate training news embeddings

import numpy as np
import torch
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/News_Recommendation_Project"
)

# Verify GPU and model
print("Model device:", model.device)
print("GPU available:", torch.cuda.is_available())

# Prepare article text
news_texts = news_df["text"].fillna("").astype(str).tolist()

# Generate embeddings
news_embeddings = model.encode(
    news_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

# Verify output
assert news_embeddings.shape == (len(news_df), 384)

# Save embeddings and their matching article information
np.save(
    PROJECT_DIR / "train_news_embeddings.npy",
    news_embeddings
)

news_df[
    ["news_id", "title", "category", "subcategory"]
].reset_index(drop=True).to_csv(
    PROJECT_DIR / "train_news_embedding_mapping.csv",
    index=False
)

print("\nEMBEDDING GENERATION COMPLETE!")
print("Embedding shape:", news_embeddings.shape)
print("Articles processed:", len(news_texts))
print("Embedding dimensions:", news_embeddings.shape[1])
print("Saved successfully to Google Drive!")

Model device: cuda:0
GPU available: True


Batches:   0%|          | 0/802 [00:00<?, ?it/s]


EMBEDDING GENERATION COMPLETE!
Embedding shape: (51282, 384)
Articles processed: 51282
Embedding dimensions: 384
Saved successfully to Google Drive!


In [34]:
# STEP 6: Reconstruct clean historical interactions

import pandas as pd

# Valid training article IDs
valid_news_ids = set(news_df["news_id"])

# Extract historical reading records
history_df = (
    train_behaviors[["user_id", "history"]]
    .assign(news_id=lambda x: x["history"].str.split())
    .explode("news_id")
    [["user_id", "news_id"]]
)

# Remove empty and invalid article IDs
history_df = history_df[
    history_df["news_id"].notna() &
    history_df["news_id"].ne("") &
    history_df["news_id"].isin(valid_news_ids)
].copy()

# Remove repeated user-article pairs
history_df = history_df.drop_duplicates(
    subset=["user_id", "news_id"]
).reset_index(drop=True)

# Save historical interactions
history_df["label"] = 1

history_df.to_pickle(
    PROJECT_DIR / "historical_interactions.pkl"
)

print("HISTORICAL INTERACTIONS READY")
print("=" * 40)
print("Unique historical interactions:", len(history_df))
print("Unique users:", history_df["user_id"].nunique())
print("Unique articles:", history_df["news_id"].nunique())

display(history_df.head())

HISTORICAL INTERACTIONS READY
Unique historical interactions: 915011
Unique users: 49108
Unique articles: 33195


,user_id,news_id,label
0,U13740,N55189,1
1,U13740,N42782,1
2,U13740,N34694,1
3,U13740,N45794,1
4,U13740,N18445,1


In [35]:
# STEP 7: Build personalized user-interest profiles

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/News_Recommendation_Project"
)

# 1. Load saved article embeddings and their mapping
news_embeddings = np.load(
    PROJECT_DIR / "train_news_embeddings.npy"
)

article_mapping = pd.read_csv(
    PROJECT_DIR / "train_news_embedding_mapping.csv"
)

# Verify embedding and article alignment
assert len(article_mapping) == news_embeddings.shape[0]
assert article_mapping["news_id"].is_unique

article_to_index = {
    news_id: index
    for index, news_id in enumerate(article_mapping["news_id"])
}

# 2. Create user and article numerical indices
user_ids = sorted(history_df["user_id"].unique())

user_to_index = {
    user_id: index
    for index, user_id in enumerate(user_ids)
}

user_indices = history_df["user_id"].map(
    user_to_index
).to_numpy()

article_indices = history_df["news_id"].map(
    article_to_index
).to_numpy()

# Verify that every historical article has an embedding
assert not pd.isna(article_indices).any()

article_indices = article_indices.astype(np.int32)

# 3. Build sparse user-article interaction matrix
interaction_matrix = csr_matrix(
    (
        np.ones(len(history_df), dtype=np.float32),
        (user_indices, article_indices)
    ),
    shape=(len(user_ids), len(article_mapping))
)

# 4. Calculate each user's average article embedding
profile_sums = interaction_matrix @ news_embeddings

interaction_counts = np.asarray(
    interaction_matrix.sum(axis=1)
).ravel()

user_profiles = (
    profile_sums /
    interaction_counts[:, np.newaxis]
)

# 5. Normalize user profiles
profile_norms = np.linalg.norm(
    user_profiles,
    axis=1,
    keepdims=True
)

user_profiles = user_profiles / np.maximum(
    profile_norms, 1e-12
)

user_profiles = user_profiles.astype(np.float32)

# 6. Save user profiles and their mapping
np.save(
    PROJECT_DIR / "user_interest_profiles.npy",
    user_profiles
)

user_mapping = pd.DataFrame({
    "user_id": user_ids,
    "interaction_count": interaction_counts.astype(int)
})

user_mapping.to_csv(
    PROJECT_DIR / "user_profile_mapping.csv",
    index=False
)

# 7. Display results
print("USER PROFILE GENERATION COMPLETE!")
print("=" * 45)

print("Users with profiles:", len(user_ids))
print("Profile matrix shape:", user_profiles.shape)
print("Embedding dimensions:", user_profiles.shape[1])
print("Total historical interactions:", len(history_df))
print("Average interactions per user:", round(interaction_counts.mean(), 2))

print("\nSaved files:")
print("1. user_interest_profiles.npy")
print("2. user_profile_mapping.csv")

display(user_mapping.head())

USER PROFILE GENERATION COMPLETE!
Users with profiles: 49108
Profile matrix shape: (49108, 384)
Embedding dimensions: 384
Total historical interactions: 915011
Average interactions per user: 18.63

Saved files:
1. user_interest_profiles.npy
2. user_profile_mapping.csv


,user_id,interaction_count
0,U100,10
1,U1000,3
2,U10001,15
3,U10003,8
4,U10008,23


In [36]:
# STEP 8: Personalized news recommendation engine

import numpy as np
import pandas as pd

# Build a fast lookup of articles already read by each user
user_seen_articles = (
    history_df.groupby("user_id")["news_id"]
    .apply(set)
    .to_dict()
)

# Recommendation function
def recommend_news(user_id, top_k=10):

    # Check whether user has an interest profile
    if user_id not in user_to_index:
        print("User not found or has no reading history.")
        return pd.DataFrame()

    # Get user's numerical profile
    user_index = user_to_index[user_id]
    user_vector = user_profiles[user_index]

    # Calculate cosine similarity scores
    # Both article embeddings and user profiles are normalized
    scores = news_embeddings @ user_vector

    # Identify articles already read
    seen_articles = user_seen_articles.get(user_id, set())

    seen_indices = [
        article_to_index[news_id]
        for news_id in seen_articles
        if news_id in article_to_index
    ]

    # Exclude previously read articles
    scores[seen_indices] = -np.inf

    # Select the highest scoring unseen articles
    available_count = np.isfinite(scores).sum()

    if available_count == 0:
        print("No unseen articles available.")
        return pd.DataFrame()

    k = min(top_k, available_count)

    top_indices = np.argpartition(
        scores, -k
    )[-k:]

    top_indices = top_indices[
        np.argsort(scores[top_indices])[::-1]
    ]

    # Prepare recommendation results
    recommendations = article_mapping.iloc[
        top_indices
    ].copy()

    recommendations["similarity_score"] = scores[top_indices]

    recommendations = recommendations.reset_index(drop=True)

    return recommendations


# Test with an existing user
test_user = user_ids[0]

print("Testing recommendation engine")
print("User ID:", test_user)

recommendations = recommend_news(
    user_id=test_user,
    top_k=10
)

display(recommendations)

Testing recommendation engine
User ID: U100


,news_id,title,category,subcategory,similarity_score
0,N64194,Mike Shildt's emotional week includes being na...,sports,baseball_mlb,0.542412
1,N33553,"Narcan, gas cards, new laws: One dad's fight a...",sports,football_nfl,0.538985
2,N33053,Tampa Bay Rays prospective pitcher posts birth...,sports,basketball_nba,0.521423
3,N2058,Fans and newcomers alike will enjoy 'Twisted T...,entertainment,entertainment-books,0.518656
4,N38035,13 Yankees who could be outta here | Aroldis C...,sports,baseball_mlb,0.507929
5,N55794,Mike Anthony: 10 years after Jasper Howard's d...,news,newsus,0.503164
6,N17509,Each Astros player's contract heading into the...,sports,baseball_mlb,0.502776
7,N44905,A few sad thoughts on Atlanta United's strange...,sports,more_sports,0.493799
8,N3397,"Epilogue: Otis Anthony, a civil rights leader ...",news,newsus,0.493719
9,N9240,MLB's most problematic contracts: Robinson Can...,sports,baseball_mlb,0.492898


In [37]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=recommendations)

https://docs.google.com/spreadsheets/d/1PD3WpvHFVATd3Q8VRB6SnbTiBk27xhkjTUg3dMTt4P4/edit#gid=0


In [38]:
# STEP 9: Verify personalized recommendations

# 1. Get the user's historical articles
user_history = history_df[
    history_df["user_id"] == test_user
]

# 2. Get previously read article IDs
seen_ids = set(user_history["news_id"])

# 3. Get recommended article IDs
recommended_ids = set(recommendations["news_id"])

# 4. Check for overlap
overlap = seen_ids.intersection(recommended_ids)

# 5. Display history and recommendation categories
history_details = article_mapping[
    article_mapping["news_id"].isin(seen_ids)
]

print("RECOMMENDATION QUALITY CHECK")
print("=" * 45)

print("Test user:", test_user)
print("Previously read articles:", len(seen_ids))
print("Recommended articles:", len(recommended_ids))
print("Previously read articles recommended:", len(overlap))

print("\nHistorical reading categories:")
display(history_details["category"].value_counts())

print("\nRecommended categories:")
display(recommendations["category"].value_counts())

print("\nPreviously read article overlap:", overlap)

assert len(overlap) == 0, "ERROR: Previously read articles found!"

print("\nSUCCESS: All recommendations are unseen articles!")

RECOMMENDATION QUALITY CHECK
Test user: U100
Previously read articles: 10
Recommended articles: 10
Previously read articles recommended: 0

Historical reading categories:


,count
category,
tv,4
lifestyle,2
sports,2
news,1
music,1



Recommended categories:


,count
category,
sports,7
news,2
entertainment,1



Previously read article overlap: set()

SUCCESS: All recommendations are unseen articles!


In [39]:
# STEP 10: Generate validation article embeddings

validation_texts = (
    val_news_df["text"]
    .fillna("")
    .astype(str)
    .tolist()
)

validation_embeddings = model.encode(
    validation_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Validation embedding shape:", validation_embeddings.shape)

# Save embeddings and article mapping
np.save(
    PROJECT_DIR / "validation_news_embeddings.npy",
    validation_embeddings
)

val_news_df[
    ["news_id", "title", "category", "subcategory"]
].reset_index(drop=True).to_csv(
    PROJECT_DIR / "validation_news_embedding_mapping.csv",
    index=False
)

print("Validation embeddings saved successfully!")

Batches:   0%|          | 0/663 [00:00<?, ?it/s]

Validation embedding shape: (42416, 384)
Validation embeddings saved successfully!


In [40]:
import numpy as np
import pandas as pd
from pathlib import Path

print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)
print("Imports successful!")

NumPy version: 2.4.6
Pandas version: 2.2.3
Imports successful!


In [41]:
# STEP 11: Combine training and validation article embeddings

# Load validation embeddings and mapping
val_embeddings = np.load(
    PROJECT_DIR / "validation_news_embeddings.npy"
)

val_mapping = pd.read_csv(
    PROJECT_DIR / "validation_news_embedding_mapping.csv"
)

# Prepare training article index
train_sources = article_mapping[
    ["news_id", "title", "category", "subcategory"]
].copy()

train_sources["_embedding_index"] = np.arange(
    len(train_sources)
)

# Prepare validation article index
val_sources = val_mapping[
    ["news_id", "title", "category", "subcategory"]
].copy()

val_sources["_embedding_index"] = (
    np.arange(len(val_sources)) + len(news_embeddings)
)

# Combine metadata
all_sources = pd.concat(
    [train_sources, val_sources],
    ignore_index=True
)

# Combine embedding matrices
all_embeddings = np.concatenate(
    [news_embeddings, val_embeddings],
    axis=0
)

# Remove duplicate article IDs
combined_mapping = (
    all_sources
    .drop_duplicates(subset="news_id", keep="first")
    .reset_index(drop=True)
)

# Align embeddings with the deduplicated mapping
combined_embeddings = all_embeddings[
    combined_mapping["_embedding_index"].to_numpy()
].astype(np.float32)

combined_mapping = combined_mapping.drop(
    columns=["_embedding_index"]
)

# Create article lookup
combined_article_to_index = {
    news_id: index
    for index, news_id in enumerate(
        combined_mapping["news_id"]
    )
}

# Save to Google Drive
np.save(
    PROJECT_DIR / "combined_news_embeddings.npy",
    combined_embeddings
)

combined_mapping.to_csv(
    PROJECT_DIR / "combined_news_mapping.csv",
    index=False
)

print("COMBINED ARTICLE INDEX READY")
print("=" * 45)
print("Training articles:", len(article_mapping))
print("Validation articles:", len(val_mapping))
print(
    "Article IDs appearing in both:",
    len(set(article_mapping["news_id"]) &
        set(val_mapping["news_id"]))
)
print("Unique combined articles:", len(combined_mapping))
print("Combined embedding shape:", combined_embeddings.shape)
print("Saved successfully!")

COMBINED ARTICLE INDEX READY
Training articles: 51282
Validation articles: 42416
Article IDs appearing in both: 28460
Unique combined articles: 65238
Combined embedding shape: (65238, 384)
Saved successfully!


In [42]:
# STEP 12: Parse validation impressions

# Give every validation impression a unique row ID
val_impressions_df = val_behaviors.copy().reset_index(drop=True)

val_impressions_df["_impression_row"] = np.arange(
    len(val_impressions_df)
)

# Split candidate article-label pairs
val_impressions_df["candidate_pairs"] = (
    val_impressions_df["impressions"].fillna("").str.split()
)

# One candidate per row
val_candidates = val_impressions_df[
    ["_impression_row", "user_id", "history", "candidate_pairs"]
].explode("candidate_pairs")

# Remove empty candidates
val_candidates = val_candidates[
    val_candidates["candidate_pairs"].notna() &
    val_candidates["candidate_pairs"].ne("")
].copy()

# Separate article ID and click label
parsed_pairs = val_candidates["candidate_pairs"].str.rsplit(
    "-", n=1, expand=True
)

val_candidates["news_id"] = parsed_pairs[0]
val_candidates["label"] = pd.to_numeric(
    parsed_pairs[1], errors="coerce"
)

# Keep valid article-label pairs
val_candidates = val_candidates[
    val_candidates["news_id"].notna() &
    val_candidates["label"].isin([0, 1])
].copy()

val_candidates["label"] = val_candidates["label"].astype(int)

# Save parsed candidates
val_candidates.to_pickle(
    PROJECT_DIR / "validation_candidates.pkl"
)

# Summary
print("VALIDATION IMPRESSIONS PARSED")
print("=" * 45)
print("Original validation impressions:", len(val_behaviors))
print("Parsed candidate records:", len(val_candidates))
print("Unique impressions:", val_candidates["_impression_row"].nunique())
print("Unique candidate articles:", val_candidates["news_id"].nunique())
print("Clicked candidates (1):", (val_candidates["label"] == 1).sum())
print("Non-clicked candidates (0):", (val_candidates["label"] == 0).sum())
print("Missing/invalid labels:", val_candidates["label"].isna().sum())

print("\nSample parsed candidates:")
display(
    val_candidates[
        ["user_id", "news_id", "label"]
    ].head(10)
)

print("\nSaved: validation_candidates.pkl")

VALIDATION IMPRESSIONS PARSED
Original validation impressions: 73152
Parsed candidate records: 2740998
Unique impressions: 73152
Unique candidate articles: 5369
Clicked candidates (1): 111383
Non-clicked candidates (0): 2629615
Missing/invalid labels: 0

Sample parsed candidates:


,user_id,news_id,label
0,U80234,N28682,0
0,U80234,N48740,0
0,U80234,N31958,1
0,U80234,N34130,0
0,U80234,N6916,0
0,U80234,N5472,0
0,U80234,N50775,0
0,U80234,N24802,0
0,U80234,N19990,0
0,U80234,N33176,0



Saved: validation_candidates.pkl


In [43]:
# STEP 13: Validate article ID coverage

# Ensure IDs use consistent string format
combined_ids = set(
    combined_mapping["news_id"].astype(str)
)

candidate_ids = set(
    val_candidates["news_id"].astype(str)
)

# Extract historical article IDs from validation histories
validation_history_ids = (
    val_behaviors["history"]
    .fillna("")
    .astype(str)
    .str.split()
    .explode()
)

validation_history_ids = set(
    validation_history_ids[
        validation_history_ids.ne("")
    ]
)

# Calculate coverage
missing_candidates = candidate_ids - combined_ids
missing_history = validation_history_ids - combined_ids

print("VALIDATION ARTICLE COVERAGE")
print("=" * 45)

print("Unique candidate article IDs:", len(candidate_ids))
print("Candidates found in embedding index:", len(candidate_ids & combined_ids))
print("Missing candidate IDs:", len(missing_candidates))

print("\nUnique historical article IDs:", len(validation_history_ids))
print("History articles found in embedding index:", len(validation_history_ids & combined_ids))
print("Missing history IDs:", len(missing_history))

print("\nCandidate coverage:",
      f"{100 * len(candidate_ids & combined_ids) / max(len(candidate_ids), 1):.2f}%")

print("History coverage:",
      f"{100 * len(validation_history_ids & combined_ids) / max(len(validation_history_ids), 1):.2f}%")

VALIDATION ARTICLE COVERAGE
Unique candidate article IDs: 5369
Candidates found in embedding index: 5369
Missing candidate IDs: 0

Unique historical article IDs: 39895
History articles found in embedding index: 37681
Missing history IDs: 2214

Candidate coverage: 100.00%
History coverage: 94.45%


In [44]:
# STEP 14: Evaluate the content-based recommender

from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

# --------------------------------------------------
# 1. Prepare article lookup
# --------------------------------------------------

article_lookup = {
    str(news_id): idx
    for idx, news_id in enumerate(
        combined_mapping["news_id"]
    )
}

# Prepare candidate data
evaluation_candidates = val_candidates.copy().reset_index(drop=True)

evaluation_candidates["_article_idx"] = (
    evaluation_candidates["news_id"]
    .astype(str)
    .map(article_lookup)
)

# Verify candidate coverage
assert evaluation_candidates["_article_idx"].notna().all(), (
    "Some candidate articles are missing embeddings!"
)

evaluation_candidates["_article_idx"] = (
    evaluation_candidates["_article_idx"].astype(int)
)

# Group candidate positions by impression
candidate_groups = evaluation_candidates.groupby(
    "_impression_row",
    sort=False
).indices

# --------------------------------------------------
# 2. Prepare validation histories
# --------------------------------------------------

validation_histories = (
    val_behaviors["history"]
    .fillna("")
    .astype(str)
    .tolist()
)

# --------------------------------------------------
# 3. Metric helper functions
# --------------------------------------------------

def calculate_ranking_metrics(scores, labels, k=10):

    ranking = np.argsort(-scores, kind="stable")
    ranked_labels = labels[ranking]

    positive_count = int(labels.sum())

    # MRR
    positive_positions = np.where(ranked_labels == 1)[0]

    mrr = (
        1.0 / (positive_positions[0] + 1)
        if len(positive_positions) > 0
        else 0.0
    )

    # Precision@K
    hits = int(ranked_labels[:k].sum())
    precision = hits / k

    # Recall@K
    recall = hits / positive_count

    # NDCG@K
    discounts = 1.0 / np.log2(
        np.arange(2, min(k, len(ranked_labels)) + 2)
    )

    dcg = np.sum(
        ranked_labels[:k] * discounts
    )

    ideal_labels = np.sort(labels)[::-1][:k]

    idcg = np.sum(
        ideal_labels * discounts[:len(ideal_labels)]
    )

    ndcg = dcg / idcg if idcg > 0 else 0.0

    return mrr, ndcg, precision, recall


# --------------------------------------------------
# 4. Evaluate every eligible impression
# --------------------------------------------------

results = []

skipped_no_history = 0
skipped_no_candidates = 0
skipped_no_positive = 0
skipped_no_negative = 0

for impression_id, history in enumerate(
    tqdm(validation_histories, desc="Evaluating impressions")
):

    candidate_positions = candidate_groups.get(
        impression_id
    )

    if candidate_positions is None:
        skipped_no_candidates += 1
        continue

    # Convert history article IDs into embedding indices
    history_ids = history.split()

    history_indices = [
        article_lookup[news_id]
        for news_id in history_ids
        if news_id in article_lookup
    ]

    if not history_indices:
        skipped_no_history += 1
        continue

    # Build normalized user-interest profile
    user_vector = combined_embeddings[
        history_indices
    ].mean(axis=0)

    norm = np.linalg.norm(user_vector)

    if norm == 0:
        skipped_no_history += 1
        continue

    user_vector = user_vector / norm

    # Retrieve impression candidates
    group = evaluation_candidates.iloc[
        candidate_positions
    ]

    candidate_indices = group[
        "_article_idx"
    ].to_numpy()

    labels = group["label"].to_numpy()

    # Calculate cosine similarity scores
    scores = (
        combined_embeddings[candidate_indices]
        @ user_vector
    )

    # AUC requires both positive and negative labels
    if labels.sum() == 0:
        skipped_no_positive += 1
        continue

    if labels.sum() == len(labels):
        skipped_no_negative += 1
        continue

    auc = roc_auc_score(labels, scores)

    mrr, ndcg, precision, recall = (
        calculate_ranking_metrics(scores, labels)
    )

    results.append({
        "impression_id": impression_id,
        "AUC": auc,
        "MRR": mrr,
        "NDCG@10": ndcg,
        "Precision@10": precision,
        "Recall@10": recall
    })


# --------------------------------------------------
# 5. Summarize evaluation
# --------------------------------------------------

evaluation_df = pd.DataFrame(results)

evaluation_df.to_csv(
    PROJECT_DIR / "baseline_evaluation_results.csv",
    index=False
)

print("\nBASELINE EVALUATION COMPLETE")
print("=" * 50)

print("Total validation impressions:", len(val_behaviors))
print("Successfully evaluated:", len(evaluation_df))

print("\nSkipped impressions:")
print("No usable history:", skipped_no_history)
print("No candidates:", skipped_no_candidates)
print("No positive clicks:", skipped_no_positive)
print("No negative candidates:", skipped_no_negative)

print("\nAVERAGE EVALUATION METRICS")
print("=" * 50)

if len(evaluation_df) > 0:
    print(
        evaluation_df[
            ["AUC", "MRR", "NDCG@10", "Precision@10", "Recall@10"]
        ].mean().round(4)
    )
else:
    print("No eligible impressions were evaluated.")

print("\nSaved: baseline_evaluation_results.csv")

Evaluating impressions:   0%|          | 0/73152 [00:00<?, ?it/s]


BASELINE EVALUATION COMPLETE
Total validation impressions: 73152
Successfully evaluated: 70938

Skipped impressions:
No usable history: 2214
No candidates: 0
No positive clicks: 0
No negative candidates: 0

AVERAGE EVALUATION METRICS
AUC             0.6469
MRR             0.3595
NDCG@10         0.4007
Precision@10    0.0849
Recall@10       0.6449
dtype: float64

Saved: baseline_evaluation_results.csv


In [45]:
# STEP 15: Popularity-based baseline evaluation

from sklearn.metrics import roc_auc_score

# 1. Calculate article popularity from training history
popularity_counts = history_df["news_id"].value_counts()

# 2. Evaluate on the same impressions as our content-based model
eligible_impressions = set(
    evaluation_df["impression_id"]
)

popularity_results = []

for impression_id in tqdm(
    sorted(eligible_impressions),
    desc="Evaluating popularity baseline"
):

    candidate_positions = candidate_groups[impression_id]

    group = evaluation_candidates.iloc[candidate_positions]

    labels = group["label"].to_numpy()

    # Popularity score for each candidate
    scores = (
        group["news_id"]
        .astype(str)
        .map(popularity_counts)
        .fillna(0)
        .to_numpy(dtype=np.float32)
    )

    # AUC
    auc = roc_auc_score(labels, scores)

    # Ranking metrics with deterministic tie-breaking
    news_ids = group["news_id"].astype(str).to_numpy()

    ranking = np.lexsort((news_ids, -scores))
    ranked_labels = labels[ranking]

    positive_positions = np.where(ranked_labels == 1)[0]

    mrr = (
        1.0 / (positive_positions[0] + 1)
        if len(positive_positions) > 0
        else 0.0
    )

    k = 10
    hits = int(ranked_labels[:k].sum())
    precision = hits / k
    recall = hits / int(labels.sum())

    discounts = 1.0 / np.log2(
        np.arange(2, min(k, len(ranked_labels)) + 2)
    )

    dcg = np.sum(
        ranked_labels[:k] * discounts
    )

    ideal_labels = np.sort(labels)[::-1][:k]

    idcg = np.sum(
        ideal_labels * discounts[:len(ideal_labels)]
    )

    ndcg = dcg / idcg if idcg > 0 else 0.0

    popularity_results.append({
        "impression_id": impression_id,
        "AUC": auc,
        "MRR": mrr,
        "NDCG@10": ndcg,
        "Precision@10": precision,
        "Recall@10": recall
    })


# 3. Save popularity results
popularity_df = pd.DataFrame(popularity_results)

popularity_df.to_csv(
    PROJECT_DIR / "popularity_evaluation_results.csv",
    index=False
)

# 4. Compare both models
content_metrics = evaluation_df[
    ["AUC", "MRR", "NDCG@10", "Precision@10", "Recall@10"]
].mean()

popularity_metrics = popularity_df[
    ["AUC", "MRR", "NDCG@10", "Precision@10", "Recall@10"]
].mean()

comparison_df = pd.DataFrame({
    "Content-Based": content_metrics,
    "Popularity-Based": popularity_metrics
})

comparison_df["Difference"] = (
    comparison_df["Content-Based"] -
    comparison_df["Popularity-Based"]
)

comparison_df.to_csv(
    PROJECT_DIR / "baseline_comparison.csv"
)

print("\nBASELINE MODEL COMPARISON")
print("=" * 60)
print("Impressions evaluated:", len(popularity_df))
print("\n", comparison_df.round(4))

print("\nSaved:")
print("- popularity_evaluation_results.csv")
print("- baseline_comparison.csv")

Evaluating popularity baseline:   0%|          | 0/70938 [00:00<?, ?it/s]


BASELINE MODEL COMPARISON
Impressions evaluated: 70938

               Content-Based  Popularity-Based  Difference
AUC                  0.6469            0.4877      0.1591
MRR                  0.3595            0.2250      0.1345
NDCG@10              0.4007            0.2705      0.1302
Precision@10         0.0849            0.0631      0.0218
Recall@10            0.6449            0.5107      0.1342

Saved:
- popularity_evaluation_results.csv
- baseline_comparison.csv


In [46]:

import pandas as pd
import numpy as np

# Load evaluation results
content_results = pd.read_csv(
    f"{PROJECT_DIR}/baseline_evaluation_results.csv"
)

popularity_results = pd.read_csv(
    f"{PROJECT_DIR}/popularity_evaluation_results.csv"
)

print("CONTENT RESULTS SHAPE:", content_results.shape)
print("POPULARITY RESULTS SHAPE:", popularity_results.shape)

print("\nCONTENT RESULT COLUMNS:")
print(content_results.columns.tolist())

print("\nPOPULARITY RESULT COLUMNS:")
print(popularity_results.columns.tolist())

print("\nCONTENT SAMPLE:")
display(content_results.head())

print("\nPOPULARITY SAMPLE:")
display(popularity_results.head())


CONTENT RESULTS SHAPE: (70938, 6)
POPULARITY RESULTS SHAPE: (70938, 6)

CONTENT RESULT COLUMNS:
['impression_id', 'AUC', 'MRR', 'NDCG@10', 'Precision@10', 'Recall@10']

POPULARITY RESULT COLUMNS:
['impression_id', 'AUC', 'MRR', 'NDCG@10', 'Precision@10', 'Recall@10']

CONTENT SAMPLE:


,impression_id,AUC,MRR,NDCG@10,Precision@10,Recall@10
0,0,0.380952,0.071429,0.000000,0.0,0.0
1,1,0.666667,0.333333,0.500000,0.1,1.0
2,2,0.227273,0.055556,0.000000,0.0,0.0
3,3,0.920000,0.333333,0.500000,0.1,1.0
4,4,0.708333,0.500000,0.580279,0.2,1.0



POPULARITY SAMPLE:


,impression_id,AUC,MRR,NDCG@10,Precision@10,Recall@10
0,0,0.428571,0.125000,0.315465,0.1,1.0
1,1,0.500000,0.500000,0.630930,0.1,1.0
2,2,0.431818,0.047619,0.000000,0.0,0.0
3,3,0.440000,0.250000,0.430677,0.1,1.0
4,4,0.500000,0.125000,0.193426,0.1,0.5


In [50]:

# STEP 16.2: Match evaluation results

metrics = [
    "AUC",
    "MRR",
    "NDCG@10",
    "Precision@10",
    "Recall@10"
]

# Match results using impression_id
paired_results = content_results.merge(
    popularity_results,
    on="impression_id",
    suffixes=("_content", "_popularity"),
    validate="one_to_one"
)

# Sort impressions consistently
paired_results = paired_results.sort_values(
    "impression_id"
).reset_index(drop=True)

# Calculate per-impression differences
for metric in metrics:
    paired_results[f"{metric}_difference"] = (
        paired_results[f"{metric}_content"] -
        paired_results[f"{metric}_popularity"]
    )

# Check matching
print("PAIRED EVALUATION DATA")
print("=" * 50)

print("Content impressions:", len(content_results))
print("Popularity impressions:", len(popularity_results))
print("Matched impressions:", len(paired_results))

print(
    "Unique matched impressions:",
    paired_results["impression_id"].nunique()
)

print("\nAVERAGE PAIRED DIFFERENCES")

difference_columns = [
    f"{metric}_difference"
    for metric in metrics
]

display(
    paired_results[difference_columns]
    .mean()
    .to_frame("Mean improvement")
)

print("\nSAMPLE PAIRED RESULTS")

display(
    paired_results[
        ["impression_id"] +
        difference_columns
    ].head(10)
)

# Save paired results
paired_results.to_csv(
    PROJECT_DIR / "paired_evaluation_results.csv",
    index=False
)

print("\nSaved: paired_evaluation_results.csv")


PAIRED EVALUATION DATA
Content impressions: 70938
Popularity impressions: 70938
Matched impressions: 70938
Unique matched impressions: 70938

AVERAGE PAIRED DIFFERENCES


,Mean improvement
AUC_difference,0.159142
MRR_difference,0.134484
NDCG@10_difference,0.130180
Precision@10_difference,0.021840
Recall@10_difference,0.134195



SAMPLE PAIRED RESULTS


,impression_id,AUC_difference,MRR_difference,NDCG@10_difference,Precision@10_difference,Recall@10_difference
0,0,-0.047619,-0.053571,-0.315465,-0.1,-1.0
1,1,0.166667,-0.166667,-0.130930,0.0,0.0
2,2,-0.204545,0.007937,0.000000,0.0,0.0
3,3,0.480000,0.083333,0.069323,0.0,0.0
4,4,0.208333,0.375000,0.386853,0.1,0.5
5,5,0.277778,0.083333,0.061296,0.0,0.0
6,6,-0.500000,-0.023810,-0.022874,0.0,0.0
7,7,0.384615,0.875000,0.419721,0.0,0.0
8,8,-0.166667,0.083333,0.069323,0.0,0.0
9,9,0.000000,-0.075000,-0.070914,0.0,0.0



Saved: paired_evaluation_results.csv


In [51]:
import inspect

print("Possible recommendation functions:")

for name, obj in globals().copy().items():
    if callable(obj) and not name.startswith("_"):
        if "recommend" in name.lower() or "article" in name.lower():
            print(name, "->", type(obj).__name__)

Possible recommendation functions:
recommend_news -> function


In [52]:
import inspect

print(inspect.getsource(recommend_news))

def recommend_news(user_id, top_k=10):

    # Check whether user has an interest profile
    if user_id not in user_to_index:
        print("User not found or has no reading history.")
        return pd.DataFrame()

    # Get user's numerical profile
    user_index = user_to_index[user_id]
    user_vector = user_profiles[user_index]

    # Calculate cosine similarity scores
    # Both article embeddings and user profiles are normalized
    scores = news_embeddings @ user_vector

    # Identify articles already read
    seen_articles = user_seen_articles.get(user_id, set())

    seen_indices = [
        article_to_index[news_id]
        for news_id in seen_articles
        if news_id in article_to_index
    ]

    # Exclude previously read articles
    scores[seen_indices] = -np.inf

    # Select the highest scoring unseen articles
    available_count = np.isfinite(scores).sum()

    if available_count == 0:
        print("No unseen articles available.")
        return pd.DataFrame()

 

In [53]:
required_variables = [
    "news_embeddings",
    "user_profiles",
    "user_to_index",
    "user_seen_articles",
    "article_mapping",
    "article_to_index"
]

for variable in required_variables:
    print(
        variable,
        "AVAILABLE" if variable in globals() else "MISSING"
    )

print("\nEmbedding shape:", news_embeddings.shape)
print("Article mapping shape:", article_mapping.shape)

news_embeddings AVAILABLE
user_profiles AVAILABLE
user_to_index AVAILABLE
user_seen_articles AVAILABLE
article_mapping AVAILABLE
article_to_index AVAILABLE

Embedding shape: (51282, 384)
Article mapping shape: (51282, 4)


In [54]:

def recommend_news_diverse(user_id, top_k=10, lambda_param=0.7):

    # Validate user
    if user_id not in user_to_index:
        print("User not found or has no reading history.")
        return pd.DataFrame()

    if not 0 <= lambda_param <= 1:
        raise ValueError("lambda_param must be between 0 and 1.")

    # Get user profile
    user_index = user_to_index[user_id]
    user_vector = user_profiles[user_index]

    # Calculate relevance scores
    relevance_scores = news_embeddings @ user_vector

    # Exclude previously read articles
    seen_articles = user_seen_articles.get(user_id, set())

    seen_indices = [
        article_to_index[news_id]
        for news_id in seen_articles
        if news_id in article_to_index
    ]

    relevance_scores[seen_indices] = -np.inf

    available_indices = np.where(
        np.isfinite(relevance_scores)
    )[0]

    if len(available_indices) == 0:
        print("No unseen articles available.")
        return pd.DataFrame()

    k = min(top_k, len(available_indices))

    # MMR selection
    selected_indices = []
    remaining_indices = available_indices.tolist()

    while len(selected_indices) < k:

        mmr_scores = []

        for candidate_idx in remaining_indices:

            relevance = relevance_scores[candidate_idx]

            if selected_indices:
                similarities = (
                    news_embeddings[selected_indices]
                    @ news_embeddings[candidate_idx]
                )

                max_similarity = np.max(similarities)

            else:
                max_similarity = 0

            mmr_score = (
                lambda_param * relevance
                - (1 - lambda_param) * max_similarity
            )

            mmr_scores.append(mmr_score)

        best_position = np.argmax(mmr_scores)
        best_index = remaining_indices[best_position]

        selected_indices.append(best_index)
        remaining_indices.remove(best_index)

    # Prepare results
    recommendations = article_mapping.iloc[
        selected_indices
    ].copy()

    recommendations["similarity_score"] = relevance_scores[
        selected_indices
    ]

    recommendations = recommendations.reset_index(drop=True)

    return recommendations


In [55]:
test_user = "U100"

diverse_recommendations = recommend_news_diverse(
    user_id=test_user,
    top_k=10,
    lambda_param=0.7
)

display(diverse_recommendations)

,news_id,title,category,subcategory,similarity_score
0,N64194,Mike Shildt's emotional week includes being na...,sports,baseball_mlb,0.542412
1,N13927,Jenelle Evans files for divorce: 'The kids and...,tv,tv-celebrity,0.483145
2,N2058,Fans and newcomers alike will enjoy 'Twisted T...,entertainment,entertainment-books,0.518656
3,N23119,Jim Carrey and Ginger Gonzaga Split After Less...,movies,movies-celebrity,0.475830
4,N33553,"Narcan, gas cards, new laws: One dad's fight a...",sports,football_nfl,0.538985
5,N33053,Tampa Bay Rays prospective pitcher posts birth...,sports,basketball_nba,0.521423
6,N44905,A few sad thoughts on Atlanta United's strange...,sports,more_sports,0.493799
7,N13435,"1 year later: DNA, reward at forefront in case...",news,newsus,0.479428
8,N3397,"Epilogue: Otis Anthony, a civil rights leader ...",news,newsus,0.493719
9,N55794,Mike Anthony: 10 years after Jasper Howard's d...,news,newsus,0.503164


In [56]:
def calculate_recommendation_diversity(recommendations):

    embeddings = news_embeddings[
        [
            article_to_index[news_id]
            for news_id in recommendations["news_id"]
            if news_id in article_to_index
        ]
    ]

    if len(embeddings) < 2:
        return np.nan

    similarity_matrix = embeddings @ embeddings.T

    upper_triangle = similarity_matrix[
        np.triu_indices(len(embeddings), k=1)
    ]

    return upper_triangle.mean()


# Generate original recommendations
original_recommendations = recommend_news(
    user_id="U100",
    top_k=10
)

# Generate diversified recommendations
diverse_recommendations = recommend_news_diverse(
    user_id="U100",
    top_k=10,
    lambda_param=0.7
)

# Compare results
comparison = pd.DataFrame({
    "Method": ["Original", "MMR Diversified"],
    "Unique Categories": [
        original_recommendations["category"].nunique(),
        diverse_recommendations["category"].nunique()
    ],
    "Average Pairwise Similarity": [
        calculate_recommendation_diversity(original_recommendations),
        calculate_recommendation_diversity(diverse_recommendations)
    ]
})

display(comparison)

,Method,Unique Categories,Average Pairwise Similarity
0,Original,3,0.359514
1,MMR Diversified,5,0.301272


In [57]:
# Save recommendations
diverse_recommendations.to_csv(
    PROJECT_DIR / "diversified_recommendations_U100.csv",
    index=False
)

comparison.to_csv(
    PROJECT_DIR / "diversity_comparison_U100.csv",
    index=False
)

print("Diversity results saved successfully!")

print("\nFiles:")
print("1. diversified_recommendations_U100.csv")
print("2. diversity_comparison_U100.csv")

Diversity results saved successfully!

Files:
1. diversified_recommendations_U100.csv
2. diversity_comparison_U100.csv


In [58]:
print("ARTICLE MAPPING COLUMNS:")
print(article_mapping.columns.tolist())

print("\nSAMPLE ARTICLES:")
display(article_mapping.head(5))

print("\nUSER PROFILE DIMENSION:")
print(user_profiles.shape)

ARTICLE MAPPING COLUMNS:
['news_id', 'title', 'category', 'subcategory']

SAMPLE ARTICLES:


,news_id,title,category,subcategory
0,N55528,"The Brands Queen Elizabeth, Prince Charles, an...",lifestyle,lifestyleroyals
1,N19639,50 Worst Habits For Belly Fat,health,weightloss
2,N61837,The Cost of Trump's Aid Freeze in the Trenches...,news,newsworld
3,N53526,I Was An NBA Wife. Here's How It Affected My M...,health,voices
4,N38324,"How to Get Rid of Skin Tags, According to a De...",health,medical



USER PROFILE DIMENSION:
(49108, 384)


In [59]:
test_user = "U100"

# Get articles previously read by this user
seen_articles = user_seen_articles.get(test_user, set())

# Find those articles in the article mapping
user_history = article_mapping[
    article_mapping["news_id"].isin(seen_articles)
].copy()

# Calculate category interests
category_interests = (
    user_history["category"]
    .value_counts()
    .rename_axis("category")
    .reset_index(name="articles_read")
)

category_interests["interest_percentage"] = (
    category_interests["articles_read"]
    / category_interests["articles_read"].sum()
    * 100
)

print("User:", test_user)
print("Historical articles found:", len(user_history))

print("\nUSER CATEGORY INTERESTS:")
display(category_interests.head(10))

User: U100
Historical articles found: 10

USER CATEGORY INTERESTS:


,category,articles_read,interest_percentage
0,tv,4,40.0
1,lifestyle,2,20.0
2,sports,2,20.0
3,news,1,10.0
4,music,1,10.0


In [60]:

def explain_recommendations(user_id, recommendations):

    # Retrieve user's reading history
    seen_articles = user_seen_articles.get(user_id, set())

    user_history = article_mapping[
        article_mapping["news_id"].isin(seen_articles)
    ]

    # Calculate category interests
    category_counts = user_history["category"].value_counts()

    total_articles = category_counts.sum()

    if total_articles == 0:
        category_percentages = {}
    else:
        category_percentages = (
            category_counts / total_articles * 100
        ).to_dict()

    # Copy recommendation results
    explained = recommendations.copy()

    explanations = []

    for _, article in explained.iterrows():

        category = article["category"]
        similarity = article["similarity_score"]

        if category in category_percentages:

            percentage = category_percentages[category]

            explanation = (
                f"You have previously read "
                f"{percentage:.1f}% of your historical articles "
                f"in the {category} category. "
                f"This article matches that interest."
            )

        else:

            explanation = (
                f"This article was selected because its content "
                f"is similar to your overall reading profile "
                f"(similarity score: {similarity:.3f})."
            )

        explanations.append(explanation)

    explained["explanation"] = explanations

    return explained


# Generate explanations
explainable_recommendations = explain_recommendations(
    user_id="U100",
    recommendations=diverse_recommendations
)

display(
    explainable_recommendations[
        [
            "news_id",
            "title",
            "category",
            "similarity_score",
            "explanation"
        ]
    ]
)


,news_id,title,category,similarity_score,explanation
0,N64194,Mike Shildt's emotional week includes being na...,sports,0.542412,You have previously read 20.0% of your histori...
1,N13927,Jenelle Evans files for divorce: 'The kids and...,tv,0.483145,You have previously read 40.0% of your histori...
2,N2058,Fans and newcomers alike will enjoy 'Twisted T...,entertainment,0.518656,This article was selected because its content ...
3,N23119,Jim Carrey and Ginger Gonzaga Split After Less...,movies,0.475830,This article was selected because its content ...
4,N33553,"Narcan, gas cards, new laws: One dad's fight a...",sports,0.538985,You have previously read 20.0% of your histori...
5,N33053,Tampa Bay Rays prospective pitcher posts birth...,sports,0.521423,You have previously read 20.0% of your histori...
6,N44905,A few sad thoughts on Atlanta United's strange...,sports,0.493799,You have previously read 20.0% of your histori...
7,N13435,"1 year later: DNA, reward at forefront in case...",news,0.479428,You have previously read 10.0% of your histori...
8,N3397,"Epilogue: Otis Anthony, a civil rights leader ...",news,0.493719,You have previously read 10.0% of your histori...
9,N55794,Mike Anthony: 10 years after Jasper Howard's d...,news,0.503164,You have previously read 10.0% of your histori...


In [61]:
# Save explainable recommendations
explainable_recommendations.to_csv(
    PROJECT_DIR / "explainable_recommendations_U100.csv",
    index=False
)

print("Explainable recommendations saved successfully!")
print("File: explainable_recommendations_U100.csv")

Explainable recommendations saved successfully!
File: explainable_recommendations_U100.csv


In [62]:
print("Available historical interaction columns:")
print(history_df.columns.tolist())

print("\nHistorical interaction sample:")
display(history_df.head())

print("\nNumber of unique articles in history:")
print(history_df["news_id"].nunique())

Available historical interaction columns:
['user_id', 'news_id', 'label']

Historical interaction sample:


,user_id,news_id,label
0,U13740,N55189,1
1,U13740,N42782,1
2,U13740,N34694,1
3,U13740,N45794,1
4,U13740,N18445,1



Number of unique articles in history:
33195


In [63]:
print("Historical interaction label distribution:")
display(history_df["label"].value_counts().to_frame("count"))

print("\nUnique labels:")
print(sorted(history_df["label"].dropna().unique()))

print("\nMissing labels:")
print(history_df["label"].isna().sum())

print("\nUnique historical users:")
print(history_df["user_id"].nunique())

Historical interaction label distribution:


,count
label,
1,915011



Unique labels:
[np.int64(1)]

Missing labels:
0

Unique historical users:
49108


In [64]:
# Calculate positive interaction counts per article
article_popularity = (
    history_df[history_df["label"] == 1]
    .groupby("news_id")
    .size()
    .reset_index(name="popularity_score")
)

# Sort by popularity
article_popularity = article_popularity.sort_values(
    "popularity_score",
    ascending=False
).reset_index(drop=True)

# Attach article information
popular_articles = article_popularity.merge(
    article_mapping,
    on="news_id",
    how="inner"
)

print("Popularity fallback created successfully!")
print("Articles ranked:", len(popular_articles))

print("\nTOP 10 POPULAR ARTICLES:")
display(popular_articles.head(10))

Popularity fallback created successfully!
Articles ranked: 33195

TOP 10 POPULAR ARTICLES:


,news_id,popularity_score,title,category,subcategory
0,N306,4747,Kevin Spacey Won't Be Charged in Sexual Assaul...,movies,movies-celebrity
1,N42620,3998,Heidi Klum's 2019 Halloween Costume Transforma...,lifestyle,lifestylebuzz
2,N45794,3283,Four flight attendants were arrested in Miami'...,news,newscrime
3,N31801,3207,Joe Biden reportedly denied Communion at a Sou...,news,newspolitics
4,N55189,3045,"'Wheel Of Fortune' Guest Delivers Hilarious, O...",tv,tvnews
5,N43142,2942,Former NBA first-round pick Jim Farmer arreste...,sports,basketball_nba
6,N29177,2915,"Miguel Cervantes' Wife Reveals Daughter, 3, 'D...",tv,tv-celebrity
7,N871,2744,"Woman, suspect dead at 'Tarzan' actor Ron Ely'...",tv,tv-celebrity
8,N16715,2644,Mitch McConnell snubbed by Elijah Cummings' pa...,news,newspolitics
9,N46392,2565,Body of missing Alabama girl found; 2 being ch...,news,newscrime


In [65]:

def recommend_cold_start(user_id=None, top_k=10):

    # Start with globally popular articles
    recommendations = popular_articles.copy()

    # Exclude previously read articles if user history exists
    if user_id is not None:
        seen_articles = user_seen_articles.get(user_id, set())

        recommendations = recommendations[
            ~recommendations["news_id"].isin(seen_articles)
        ]

    # Select top popular unseen articles
    recommendations = recommendations.head(top_k).copy()

    if recommendations.empty:
        print("No recommendations available.")
        return pd.DataFrame()

    recommendations = recommendations.reset_index(drop=True)

    return recommendations[
        [
            "news_id",
            "title",
            "category",
            "subcategory",
            "popularity_score"
        ]
    ]


In [66]:
cold_start_results = recommend_cold_start(
    user_id="NEW_USER_001",
    top_k=10
)

print("COLD-START RECOMMENDATIONS")
display(cold_start_results)

COLD-START RECOMMENDATIONS


,news_id,title,category,subcategory,popularity_score
0,N306,Kevin Spacey Won't Be Charged in Sexual Assaul...,movies,movies-celebrity,4747
1,N42620,Heidi Klum's 2019 Halloween Costume Transforma...,lifestyle,lifestylebuzz,3998
2,N45794,Four flight attendants were arrested in Miami'...,news,newscrime,3283
3,N31801,Joe Biden reportedly denied Communion at a Sou...,news,newspolitics,3207
4,N55189,"'Wheel Of Fortune' Guest Delivers Hilarious, O...",tv,tvnews,3045
5,N43142,Former NBA first-round pick Jim Farmer arreste...,sports,basketball_nba,2942
6,N29177,"Miguel Cervantes' Wife Reveals Daughter, 3, 'D...",tv,tv-celebrity,2915
7,N871,"Woman, suspect dead at 'Tarzan' actor Ron Ely'...",tv,tv-celebrity,2744
8,N16715,Mitch McConnell snubbed by Elijah Cummings' pa...,news,newspolitics,2644
9,N46392,Body of missing Alabama girl found; 2 being ch...,news,newscrime,2565


In [67]:
cold_start_results.to_csv(
    PROJECT_DIR / "cold_start_recommendations.csv",
    index=False
)

popular_articles.to_csv(
    PROJECT_DIR / "article_popularity_ranking.csv",
    index=False
)

print("Cold-start results saved successfully!")
print("1. cold_start_recommendations.csv")
print("2. article_popularity_ranking.csv")

Cold-start results saved successfully!
1. cold_start_recommendations.csv
2. article_popularity_ranking.csv


In [68]:
def recommend_news_unified(user_id, top_k=10, lambda_param=0.7):

    # Use personalized recommendations when a user profile exists
    if user_id in user_to_index:

        recommendations = recommend_news_diverse(
            user_id=user_id,
            top_k=top_k,
            lambda_param=lambda_param
        )

        if recommendations.empty:
            return recommendations

        recommendations["recommendation_type"] = "personalized_diverse"

    # Otherwise, use popularity-based cold-start recommendations
    else:

        recommendations = recommend_cold_start(
            user_id=user_id,
            top_k=top_k
        )

        if recommendations.empty:
            return recommendations

        recommendations["recommendation_type"] = "cold_start_popularity"

    return recommendations.reset_index(drop=True)


print("Unified recommendation function created successfully!")

Unified recommendation function created successfully!


In [69]:
# Test 1: Existing user
existing_user_results = recommend_news_unified(
    user_id="U100",
    top_k=10
)

print("EXISTING USER RECOMMENDATIONS")
display(existing_user_results)

# Test 2: New user
new_user_results = recommend_news_unified(
    user_id="NEW_USER_001",
    top_k=10
)

print("NEW USER RECOMMENDATIONS")
display(new_user_results)

# Verify recommendation types
print("\nRecommendation types:")
print("Existing user:", existing_user_results["recommendation_type"].unique())
print("New user:", new_user_results["recommendation_type"].unique())

EXISTING USER RECOMMENDATIONS


,news_id,title,category,subcategory,similarity_score,recommendation_type
0,N64194,Mike Shildt's emotional week includes being na...,sports,baseball_mlb,0.542412,personalized_diverse
1,N13927,Jenelle Evans files for divorce: 'The kids and...,tv,tv-celebrity,0.483145,personalized_diverse
2,N2058,Fans and newcomers alike will enjoy 'Twisted T...,entertainment,entertainment-books,0.518656,personalized_diverse
3,N23119,Jim Carrey and Ginger Gonzaga Split After Less...,movies,movies-celebrity,0.475830,personalized_diverse
4,N33553,"Narcan, gas cards, new laws: One dad's fight a...",sports,football_nfl,0.538985,personalized_diverse
5,N33053,Tampa Bay Rays prospective pitcher posts birth...,sports,basketball_nba,0.521423,personalized_diverse
6,N44905,A few sad thoughts on Atlanta United's strange...,sports,more_sports,0.493799,personalized_diverse
7,N13435,"1 year later: DNA, reward at forefront in case...",news,newsus,0.479428,personalized_diverse
8,N3397,"Epilogue: Otis Anthony, a civil rights leader ...",news,newsus,0.493719,personalized_diverse
9,N55794,Mike Anthony: 10 years after Jasper Howard's d...,news,newsus,0.503164,personalized_diverse


NEW USER RECOMMENDATIONS


,news_id,title,category,subcategory,popularity_score,recommendation_type
0,N306,Kevin Spacey Won't Be Charged in Sexual Assaul...,movies,movies-celebrity,4747,cold_start_popularity
1,N42620,Heidi Klum's 2019 Halloween Costume Transforma...,lifestyle,lifestylebuzz,3998,cold_start_popularity
2,N45794,Four flight attendants were arrested in Miami'...,news,newscrime,3283,cold_start_popularity
3,N31801,Joe Biden reportedly denied Communion at a Sou...,news,newspolitics,3207,cold_start_popularity
4,N55189,"'Wheel Of Fortune' Guest Delivers Hilarious, O...",tv,tvnews,3045,cold_start_popularity
5,N43142,Former NBA first-round pick Jim Farmer arreste...,sports,basketball_nba,2942,cold_start_popularity
6,N29177,"Miguel Cervantes' Wife Reveals Daughter, 3, 'D...",tv,tv-celebrity,2915,cold_start_popularity
7,N871,"Woman, suspect dead at 'Tarzan' actor Ron Ely'...",tv,tv-celebrity,2744,cold_start_popularity
8,N16715,Mitch McConnell snubbed by Elijah Cummings' pa...,news,newspolitics,2644,cold_start_popularity
9,N46392,Body of missing Alabama girl found; 2 being ch...,news,newscrime,2565,cold_start_popularity



Recommendation types:
Existing user: ['personalized_diverse']
New user: ['cold_start_popularity']


In [70]:
# Check existing user's recommendations
existing_seen = user_seen_articles.get("U100", set())

existing_seen_count = existing_user_results["news_id"].isin(
    existing_seen
).sum()

existing_duplicates = existing_user_results["news_id"].duplicated().sum()

# Check new user's recommendations
new_duplicates = new_user_results["news_id"].duplicated().sum()

popularity_ordered = (
    new_user_results["popularity_score"].is_monotonic_decreasing
)

print("EXISTING USER CHECKS")
print("Recommended articles already read:", existing_seen_count)
print("Duplicate recommendations:", existing_duplicates)

print("\nNEW USER CHECKS")
print("Duplicate recommendations:", new_duplicates)
print("Popularity ranking is descending:", popularity_ordered)

# Save both outputs
existing_user_results.to_csv(
    PROJECT_DIR / "unified_existing_user_recommendations.csv",
    index=False
)

new_user_results.to_csv(
    PROJECT_DIR / "unified_new_user_recommendations.csv",
    index=False
)

print("\nBoth unified recommendation results saved successfully!")

EXISTING USER CHECKS
Recommended articles already read: 0
Duplicate recommendations: 0

NEW USER CHECKS
Duplicate recommendations: 0
Popularity ranking is descending: True

Both unified recommendation results saved successfully!


In [71]:
# Select 5 users reproducibly
rng = np.random.default_rng(42)
sample_users = rng.choice(
    list(user_to_index.keys()),
    size=min(5, len(user_to_index)),
    replace=False
)

diversity_results = []

for user_id in sample_users:

    # Original recommendations
    original = recommend_news(user_id, top_k=10)

    # Diversity-aware recommendations
    diverse = recommend_news_diverse(
        user_id,
        top_k=10,
        lambda_param=0.7
    )

    for method, recommendations in [
        ("Original", original),
        ("MMR Diversified", diverse)
    ]:

        if len(recommendations) < 2:
            continue

        # Count unique categories
        unique_categories = recommendations["category"].nunique()

        # Calculate average pairwise cosine similarity
        indices = [
            article_to_index[news_id]
            for news_id in recommendations["news_id"]
            if news_id in article_to_index
        ]

        vectors = news_embeddings[indices]
        similarity_matrix = vectors @ vectors.T

        upper_triangle = np.triu_indices(
            len(vectors),
            k=1
        )

        average_similarity = similarity_matrix[
            upper_triangle
        ].mean()

        diversity_results.append({
            "user_id": user_id,
            "method": method,
            "unique_categories": unique_categories,
            "average_pairwise_similarity": average_similarity
        })

# Create results table
diversity_df = pd.DataFrame(diversity_results)

print("MULTI-USER DIVERSITY RESULTS")
display(diversity_df)

# Calculate average results by method
diversity_summary = (
    diversity_df
    .groupby("method")[
        ["unique_categories", "average_pairwise_similarity"]
    ]
    .mean()
    .reset_index()
)

print("\nAVERAGE DIVERSITY SUMMARY")
display(diversity_summary)

# Save results
diversity_df.to_csv(
    PROJECT_DIR / "multi_user_diversity_results.csv",
    index=False
)

diversity_summary.to_csv(
    PROJECT_DIR / "multi_user_diversity_summary.csv",
    index=False
)

print("\nMulti-user diversity results saved successfully!")

MULTI-USER DIVERSITY RESULTS


,user_id,method,unique_categories,average_pairwise_similarity
0,U75581,Original,6,0.390893
1,U75581,MMR Diversified,9,0.284270
2,U47368,Original,3,0.345298
3,U47368,MMR Diversified,3,0.297493
4,U65654,Original,4,0.381420
5,U65654,MMR Diversified,5,0.315612
6,U17489,Original,5,0.521813
7,U17489,MMR Diversified,7,0.356445
8,U46807,Original,5,0.369486
9,U46807,MMR Diversified,7,0.280838



AVERAGE DIVERSITY SUMMARY


,method,unique_categories,average_pairwise_similarity
0,MMR Diversified,6.2,0.306932
1,Original,4.6,0.401782



Multi-user diversity results saved successfully!


In [72]:
lambda_values = [0.5, 0.7, 0.9]
lambda_results = []

for user_id in sample_users:

    for lambda_value in lambda_values:

        recommendations = recommend_news_diverse(
            user_id=user_id,
            top_k=10,
            lambda_param=lambda_value
        )

        if len(recommendations) < 2:
            continue

        indices = [
            article_to_index[news_id]
            for news_id in recommendations["news_id"]
            if news_id in article_to_index
        ]

        vectors = news_embeddings[indices]
        similarity_matrix = vectors @ vectors.T

        upper_triangle = np.triu_indices(
            len(vectors),
            k=1
        )

        average_similarity = similarity_matrix[
            upper_triangle
        ].mean()

        lambda_results.append({
            "user_id": user_id,
            "lambda": lambda_value,
            "unique_categories": recommendations["category"].nunique(),
            "average_pairwise_similarity": average_similarity,
            "average_relevance_score": recommendations["similarity_score"].mean()
        })

lambda_df = pd.DataFrame(lambda_results)

lambda_summary = (
    lambda_df
    .groupby("lambda")[
        [
            "unique_categories",
            "average_pairwise_similarity",
            "average_relevance_score"
        ]
    ]
    .mean()
    .reset_index()
)

print("MMR PARAMETER COMPARISON")
display(lambda_summary)

lambda_df.to_csv(
    PROJECT_DIR / "mmr_lambda_user_results.csv",
    index=False
)

lambda_summary.to_csv(
    PROJECT_DIR / "mmr_lambda_summary.csv",
    index=False
)

print("MMR parameter results saved successfully!")

MMR PARAMETER COMPARISON


,lambda,unique_categories,average_pairwise_similarity,average_relevance_score
0,0.5,6.0,0.194555,0.447657
1,0.7,6.2,0.306932,0.519412
2,0.9,5.4,0.374672,0.543422


MMR parameter results saved successfully!


In [73]:
import json

recommendation_config = {
    "recommendation_method": "Content-based filtering",
    "embedding_model": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    "embedding_dimension": 384,
    "personalized_recommendations": "MMR diversification",
    "mmr_lambda": 0.7,
    "default_top_k": 10,
    "cold_start_method": "Popularity-based ranking",
    "seen_article_filter": True
}

config_path = PROJECT_DIR / "recommendation_system_config.json"

with open(config_path, "w") as file:
    json.dump(recommendation_config, file, indent=4)

# Reload and verify the saved configuration
with open(config_path, "r") as file:
    loaded_config = json.load(file)

print("CONFIGURATION SAVED AND VERIFIED")
print(json.dumps(loaded_config, indent=4))

print("\nConfiguration file:", config_path)

CONFIGURATION SAVED AND VERIFIED
{
    "recommendation_method": "Content-based filtering",
    "embedding_model": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    "embedding_dimension": 384,
    "personalized_recommendations": "MMR diversification",
    "mmr_lambda": 0.7,
    "default_top_k": 10,
    "cold_start_method": "Popularity-based ranking",
    "seen_article_filter": true
}

Configuration file: /content/drive/MyDrive/News_Recommendation_Project/recommendation_system_config.json


In [74]:
import os

print("PROJECT DIRECTORY")
print(PROJECT_DIR)

files = sorted(os.listdir(PROJECT_DIR))

print("\nSAVED PROJECT FILES")
for filename in files:
    filepath = PROJECT_DIR / filename

    if filepath.is_file():
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        print(f"{filename} — {size_mb:.2f} MB")
    else:
        print(f"{filename}/ — directory")

print("\nTotal files and directories:", len(files))

PROJECT DIRECTORY
/content/drive/MyDrive/News_Recommendation_Project

SAVED PROJECT FILES
article_popularity_ranking.csv — 3.04 MB
baseline_comparison.csv — 0.00 MB
baseline_evaluation_results.csv — 3.50 MB
cold_start_recommendations.csv — 0.00 MB
combined_news_embeddings.npy — 95.56 MB
combined_news_mapping.csv — 5.82 MB
diversified_recommendations_U100.csv — 0.00 MB
diversity_comparison_U100.csv — 0.00 MB
explainable_recommendations_U100.csv — 0.00 MB
historical_interactions.pkl — 19.20 MB
mmr_lambda_summary.csv — 0.00 MB
mmr_lambda_user_results.csv — 0.00 MB
multi_user_diversity_results.csv — 0.00 MB
multi_user_diversity_summary.csv — 0.00 MB
paired_evaluation_results.csv — 10.22 MB
popularity_evaluation_results.csv — 3.53 MB
raw_data/ — directory
recommendation_system_config.json — 0.00 MB
statistical_validation_results.csv — 0.00 MB
train_behaviors.pkl — 62.41 MB
train_news_clean.pkl — 52.87 MB
train_news_embedding_mapping.csv — 4.58 MB
train_news_embeddings.npy — 75.12 MB
unified

In [75]:
import json
import numpy as np
import pandas as pd
import pickle

# 1. Reload configuration
with open(PROJECT_DIR / "recommendation_system_config.json", "r") as f:
    saved_config = json.load(f)

# 2. Reload embeddings without copying the full arrays into memory
saved_news_embeddings = np.load(
    PROJECT_DIR / "combined_news_embeddings.npy",
    mmap_mode="r"
)

saved_user_profiles = np.load(
    PROJECT_DIR / "user_interest_profiles.npy",
    mmap_mode="r"
)

# 3. Reload mappings
saved_news_mapping = pd.read_csv(
    PROJECT_DIR / "combined_news_mapping.csv"
)

saved_user_mapping = pd.read_csv(
    PROJECT_DIR / "user_profile_mapping.csv"
)

# 4. Reload historical interactions
with open(PROJECT_DIR / "historical_interactions.pkl", "rb") as f:
    saved_history = pickle.load(f)

# 5. Reload evaluation summaries
saved_baseline_comparison = pd.read_csv(
    PROJECT_DIR / "baseline_comparison.csv"
)

saved_statistical_results = pd.read_csv(
    PROJECT_DIR / "statistical_validation_results.csv"
)

# 6. Display verification results
print("RELOAD VERIFICATION")
print("-------------------")

print("Configuration loaded:", bool(saved_config))
print("News embeddings shape:", saved_news_embeddings.shape)
print("User profiles shape:", saved_user_profiles.shape)
print("News mapping rows:", len(saved_news_mapping))
print("User mapping rows:", len(saved_user_mapping))
print("Historical interactions:", len(saved_history))

print("\nBaseline comparison rows:", len(saved_baseline_comparison))
print("Statistical validation rows:", len(saved_statistical_results))

# 7. Check alignment
print("\nALIGNMENT CHECKS")
print(
    "News embeddings and mapping aligned:",
    saved_news_embeddings.shape[0] == len(saved_news_mapping)
)
print(
    "User profiles and mapping aligned:",
    saved_user_profiles.shape[0] == len(saved_user_mapping)
)

print("\nReload verification completed!")

RELOAD VERIFICATION
-------------------
Configuration loaded: True
News embeddings shape: (65238, 384)
User profiles shape: (49108, 384)
News mapping rows: 65238
User mapping rows: 49108
Historical interactions: 915011

Baseline comparison rows: 5
Statistical validation rows: 5

ALIGNMENT CHECKS
News embeddings and mapping aligned: True
User profiles and mapping aligned: True

Reload verification completed!


In [76]:
# Reload saved recommendation outputs
saved_existing = pd.read_csv(
    PROJECT_DIR / "unified_existing_user_recommendations.csv"
)

saved_new = pd.read_csv(
    PROJECT_DIR / "unified_new_user_recommendations.csv"
)

# Verify results
print("FINAL OUTPUT VERIFICATION")
print("-------------------------")

print("Existing user recommendations:", len(saved_existing))
print("New user recommendations:", len(saved_new))

print(
    "Existing user method:",
    saved_existing["recommendation_type"].unique()
)

print(
    "New user method:",
    saved_new["recommendation_type"].unique()
)

print(
    "Existing user has duplicate articles:",
    saved_existing["news_id"].duplicated().any()
)

print(
    "New user has duplicate articles:",
    saved_new["news_id"].duplicated().any()
)

assert len(saved_existing) == 10
assert len(saved_new) == 10
assert not saved_existing["news_id"].duplicated().any()
assert not saved_new["news_id"].duplicated().any()

print("\nFINAL OUTPUT VERIFICATION PASSED!")

FINAL OUTPUT VERIFICATION
-------------------------
Existing user recommendations: 10
New user recommendations: 10
Existing user method: ['personalized_diverse']
New user method: ['cold_start_popularity']
Existing user has duplicate articles: False
New user has duplicate articles: False

FINAL OUTPUT VERIFICATION PASSED!
